# SHS27k PPI: GIN vs dpdGAT -- Colab, sequential, no threading

Same pipeline as the WSL version, rebuilt to run entirely on Colab: no multiprocessing, no
background processes, no local environment. Each of the 30 jobs (2 models x 3 splits x 5 seeds)
runs one after another, and every single result is written to a CSV in your Google Drive the
moment it finishes -- so even if the Colab session disconnects partway through, nothing already
completed is lost. Just re-run the loop cell and it'll skip jobs already in the CSV.

**Before running:** upload `protein.SHS27k.sequences.dictionary.pro3.tsv`,
`protein.actions.SHS27k.STRING.pro2.txt`, and `emb_esm2_t12_35M_UR50D.pt` into a folder in your
Google Drive (e.g. `MyDrive/ppi_project/`), then set `DRIVE_DIR` in the config cell below to match.

## 1. Mount Drive + install torch_geometric

Colab has PyTorch preinstalled but not PyG -- this is the one thing we install.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q torch_geometric

## 2. Config -- set this to wherever you uploaded the three files

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/ppi_project"  # <-- change this if your folder is named differently

SEQ_FILE = os.path.join(DRIVE_DIR, "protein.SHS27k.sequences.dictionary.pro3.tsv")
ACTIONS_FILE = os.path.join(DRIVE_DIR, "protein.actions.SHS27k.STRING.pro2.txt")
EMB_FILE = os.path.join(DRIVE_DIR, "emb_esm2_t12_35M_UR50D.pt")
RESULTS_CSV = os.path.join(DRIVE_DIR, "grid_results.csv")  # also saved to Drive, survives disconnects

for f in [SEQ_FILE, ACTIONS_FILE, EMB_FILE]:
    print(f, "FOUND" if os.path.exists(f) else "MISSING -- check the filename/path")

## 3. Setup, imports, seeds

In [ ]:
import random
import csv
import time
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score

MODES = ["reaction", "binding", "catalysis", "activation", "inhibition", "ptmod", "expression"]
MODE_TO_IDX = {m: i for i, m in enumerate(MODES)}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

## 4. Load proteins, embeddings, and collapse the interaction file into labeled pairs

Same logic as before: 6,660 unique unordered pairs expected, ~58% multi-label.

In [ ]:
seq_df = pd.read_csv(SEQ_FILE, sep="\t", header=None, names=["protein_id", "sequence"])
protein_ids = seq_df["protein_id"].tolist()
print(f"Loaded {len(protein_ids)} proteins")

emb_data = torch.load(EMB_FILE, map_location="cpu")
emb_ids = emb_data["ids"]      # confirmed keys from the earlier debug print
emb_tensor = emb_data["emb"]
print(f"Embeddings: {tuple(emb_tensor.shape)}, no NaNs: {not torch.isnan(emb_tensor).any().item()}")

id_to_emb_row = {pid: i for i, pid in enumerate(emb_ids)}
pid_to_node = {pid: i for i, pid in enumerate(protein_ids)}
node_features = torch.zeros(len(protein_ids), emb_tensor.shape[1])
for pid, node_idx in pid_to_node.items():
    if pid in id_to_emb_row:
        node_features[node_idx] = emb_tensor[id_to_emb_row[pid]]

actions_df = pd.read_csv(ACTIONS_FILE, sep=r"\s+")
pair_labels = defaultdict(lambda: np.zeros(len(MODES), dtype=np.float32))
for row in actions_df.itertuples(index=False):
    a, b, mode = row.item_id_a, row.item_id_b, row.mode
    if a not in pid_to_node or b not in pid_to_node:
        continue
    key = tuple(sorted((a, b)))
    pair_labels[key][MODE_TO_IDX[mode]] = 1.0

pairs = list(pair_labels.keys())
labels = np.stack([pair_labels[p] for p in pairs])
print(f"Unique unordered pairs: {len(pairs)}")                          # expect 6,660
print(f"Multi-label fraction: {(labels.sum(axis=1) > 1).mean():.2%}")   # expect ~58%

## 5. Seeded Random / BFS / DFS splits

In [ ]:
def build_adjacency(pairs):
    adj = defaultdict(list)
    for idx, (a, b) in enumerate(pairs):
        adj[a].append((b, idx))
        adj[b].append((a, idx))
    return adj

adj = build_adjacency(pairs)
node_degree = {n: len(v) for n, v in adj.items()}


def random_split(pairs, seed, ratios=(0.6, 0.2, 0.2)):
    rng = np.random.RandomState(seed)
    idx = np.arange(len(pairs))
    rng.shuffle(idx)
    n = len(pairs)
    n_train, n_valid = int(n * ratios[0]), int(n * ratios[1])
    return idx[:n_train], idx[n_train:n_train + n_valid], idx[n_train + n_valid:]


def _walk_split(pairs, adj, seed, ratios, mode="bfs"):
    rng = random.Random(seed)
    n_edges = len(pairs)
    target_train = int(n_edges * ratios[0])
    target_valid = int(n_edges * ratios[1])

    low_degree_pool = sorted(node_degree, key=lambda n: node_degree[n])[:max(1, len(node_degree) // 10)]
    start_node = rng.choice(low_degree_pool)

    visited_edges, visited_nodes = set(), {start_node}
    frontier = deque([start_node]) if mode == "bfs" else [start_node]
    order = []

    while frontier and len(visited_edges) < n_edges:
        node = frontier.popleft() if mode == "bfs" else frontier.pop()
        neighbors = list(adj[node])
        rng.shuffle(neighbors)
        for nbr, eidx in neighbors:
            if eidx not in visited_edges:
                visited_edges.add(eidx)
                order.append(eidx)
            if nbr not in visited_nodes:
                visited_nodes.add(nbr)
                frontier.append(nbr)

    remaining = [i for i in range(n_edges) if i not in visited_edges]
    rng.shuffle(remaining)
    order.extend(remaining)

    train_idx = np.array(order[:target_train])
    valid_idx = np.array(order[target_train:target_train + target_valid])
    test_idx = np.array(order[target_train + target_valid:])
    return train_idx, valid_idx, test_idx


SPLIT_FNS = {
    "random": lambda seed: random_split(pairs, seed),
    "bfs": lambda seed: _walk_split(pairs, adj, seed, (0.6, 0.2, 0.2), "bfs"),
    "dfs": lambda seed: _walk_split(pairs, adj, seed, (0.6, 0.2, 0.2), "dfs"),
}

for name, fn in SPLIT_FNS.items():
    tr, va, te = fn(42)
    print(name, len(tr), len(va), len(te))

## 6. PyG Data + both models (GIN baseline, dpdGAT from TRGH-PPI)

In [ ]:
from torch_geometric.data import Data
from torch_geometric.nn import GINConv
from torch_geometric.utils import softmax, scatter


def make_pyg_data(pairs, labels, train_idx, valid_idx, test_idx):
    src = torch.tensor([pid_to_node[a] for a, b in pairs], dtype=torch.long)
    dst = torch.tensor([pid_to_node[b] for a, b in pairs], dtype=torch.long)
    y = torch.tensor(labels, dtype=torch.float32)

    train_mask = torch.zeros(len(pairs), dtype=torch.bool)
    valid_mask = torch.zeros(len(pairs), dtype=torch.bool)
    test_mask = torch.zeros(len(pairs), dtype=torch.bool)
    train_mask[train_idx] = True
    valid_mask[valid_idx] = True
    test_mask[test_idx] = True

    return Data(x=node_features, edge_src=src, edge_dst=dst, y=y,
                train_mask=train_mask, valid_mask=valid_mask, test_mask=test_mask)


def train_edge_index(data):
    m = data.train_mask
    s, d = data.edge_src[m], data.edge_dst[m]
    return torch.stack([torch.cat([s, d]), torch.cat([d, s])])


class GINEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_layers=3, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        dims = [in_dim] + [hidden_dim] * num_layers
        for i in range(num_layers):
            mlp = nn.Sequential(nn.Linear(dims[i], hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x


class PPIClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, num_layers=3, num_classes=7, dropout=0.2):
        super().__init__()
        self.encoder = GINEncoder(in_dim, hidden_dim, num_layers, dropout)
        self.head = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                                   nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes))

    def forward(self, x, edge_index_mp, pair_src, pair_dst):
        h = self.encoder(x, edge_index_mp)
        return self.head(h[pair_src] * h[pair_dst])


class DpdGATLayer(nn.Module):
    """TRGH-PPI Eqs 12-14. Aggregation is attention-weighted -- see the earlier discussion on
    Eq 14 as printed in the paper omitting the attention term, likely a typo."""
    def __init__(self, in_dim, hidden_dim, heads=2, eps_init=0.0, dropout=0.0):
        super().__init__()
        assert hidden_dim % heads == 0
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.hidden_dim = hidden_dim
        self.W = nn.Linear(in_dim, hidden_dim, bias=False)
        self.att = nn.Parameter(torch.empty(heads, self.head_dim))
        nn.init.xavier_uniform_(self.att)
        self.leaky_relu = nn.LeakyReLU(0.2)
        self.self_proj = nn.Linear(in_dim, hidden_dim)
        self.eps = nn.Parameter(torch.tensor(eps_init))
        self.mlp = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        src, dst = edge_index[0], edge_index[1]
        N = x.size(0)
        z = self.W(x).view(N, self.heads, self.head_dim)
        z_src, z_dst = z[src], z[dst]
        raw = (z_dst * z_src) / (self.head_dim ** 0.5)
        e = (self.leaky_relu(raw) * self.att).sum(dim=-1)
        gamma = softmax(e, dst, num_nodes=N)
        weighted = z_src * gamma.unsqueeze(-1)
        agg = scatter(weighted, dst, dim=0, dim_size=N, reduce="sum").reshape(N, self.hidden_dim)
        self_term = self.self_proj(x)
        combined = (1 + self.eps) * self_term + agg
        out = self.bn(F.relu(self.mlp(combined)))
        return F.dropout(out, p=self.dropout, training=self.training)


class DpdGATEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=1024, num_layers=3, heads=2, dropout=0.0):
        super().__init__()
        dims = [in_dim] + [hidden_dim] * num_layers
        self.layers = nn.ModuleList([DpdGATLayer(dims[i], hidden_dim, heads=heads, dropout=dropout)
                                      for i in range(num_layers)])

    def forward(self, x, edge_index):
        for layer in self.layers:
            x = layer(x, edge_index)
        return x


class PPIClassifierDpdGAT(nn.Module):
    """Eq 15: y_hat_ij = FC(x_i * x_j) -- single linear layer, matching the paper."""
    def __init__(self, in_dim, hidden_dim=1024, num_layers=3, heads=2, num_classes=7, dropout=0.0):
        super().__init__()
        self.encoder = DpdGATEncoder(in_dim, hidden_dim, num_layers, heads, dropout)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index_mp, pair_src, pair_dst):
        h = self.encoder(x, edge_index_mp)
        return self.head(h[pair_src] * h[pair_dst])


def build_model(model_type, in_dim, hidden_dim, num_layers, heads=2, num_classes=7, dropout=0.2):
    if model_type == "gin":
        return PPIClassifier(in_dim, hidden_dim, num_layers, num_classes, dropout)
    elif model_type == "dpdgat":
        return PPIClassifierDpdGAT(in_dim, hidden_dim, num_layers, heads, num_classes, dropout=0.0)
    raise ValueError(model_type)

## 7. Training loop

In [ ]:
def run_experiment(model_type, split_name, seed, epochs, lr=1e-3, weight_decay=1e-4,
                    hidden_dim=256, num_layers=3, heads=2, device="cpu"):
    set_seed(seed)
    train_idx, valid_idx, test_idx = SPLIT_FNS[split_name](seed)
    data = make_pyg_data(pairs, labels, train_idx, valid_idx, test_idx).to(device)
    mp_edge_index = train_edge_index(data).to(device)

    model = build_model(model_type, in_dim=node_features.shape[1], hidden_dim=hidden_dim,
                         num_layers=num_layers, heads=heads).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_valid_f1, best_state = -1.0, None
    check_every = max(1, epochs // 20)

    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        logits = model(data.x, mp_edge_index, data.edge_src[data.train_mask], data.edge_dst[data.train_mask])
        loss = criterion(logits, data.y[data.train_mask])
        loss.backward()
        opt.step()

        if epoch % check_every == 0 or epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                val_logits = model(data.x, mp_edge_index, data.edge_src[data.valid_mask], data.edge_dst[data.valid_mask])
                val_pred = (torch.sigmoid(val_logits) > 0.5).float()
                val_f1 = f1_score(data.y[data.valid_mask].numpy(), val_pred.numpy(), average="micro", zero_division=0)
            if val_f1 > best_valid_f1:
                best_valid_f1 = val_f1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_logits = model(data.x, mp_edge_index, data.edge_src[data.test_mask], data.edge_dst[data.test_mask])
        test_pred = (torch.sigmoid(test_logits) > 0.5).float()
        test_f1 = f1_score(data.y[data.test_mask].numpy(), test_pred.numpy(), average="micro", zero_division=0)

    return {"model": model_type, "split": split_name, "seed": seed,
            "best_valid_f1": best_valid_f1, "test_micro_f1": test_f1}

## 8. Sequential grid -- no threading, no multiprocessing

Runs one job at a time. Writes each result to `RESULTS_CSV` in Drive immediately after it
finishes, and **skips any job already present in the CSV** -- so if Colab disconnects partway
through, just re-run this exact cell and it picks up where it left off instead of starting over.

In [ ]:
MODEL_TYPES = ["gin", "dpdgat"]
SPLIT_NAMES = ["random", "bfs", "dfs"]
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = 200

jobs = [(m, s, seed) for m in MODEL_TYPES for s in SPLIT_NAMES for seed in SEEDS]

# Resume support: load any already-completed jobs from a previous run
if os.path.exists(RESULTS_CSV):
    done_df = pd.read_csv(RESULTS_CSV)
    done_keys = set(zip(done_df["model"], done_df["split"], done_df["seed"]))
    print(f"Resuming -- {len(done_keys)} jobs already completed, will be skipped")
else:
    done_keys = set()
    with open(RESULTS_CSV, "w", newline="") as f:
        csv.writer(f).writerow(["model", "split", "seed", "best_valid_f1", "test_micro_f1"])

t0 = time.time()
for i, (model_type, split_name, seed) in enumerate(jobs, 1):
    if (model_type, split_name, seed) in done_keys:
        continue

    hidden_dim = 256 if model_type == "gin" else 1024
    res = run_experiment(model_type, split_name, seed, epochs=EPOCHS, hidden_dim=hidden_dim)

    with open(RESULTS_CSV, "a", newline="") as f:
        csv.writer(f).writerow([res["model"], res["split"], res["seed"],
                                 res["best_valid_f1"], res["test_micro_f1"]])

    elapsed = time.time() - t0
    print(f"[{i}/{len(jobs)}] {res}  ({elapsed:.0f}s elapsed)")

print("All jobs complete (or already were).")

## 9. Compare against TRGH-PPI's reported numbers (Table IV, SHS27k)

In [ ]:
results_df = pd.read_csv(RESULTS_CSV)
summary = results_df.groupby(["model", "split"])["test_micro_f1"].agg(["mean", "std", "count"])
print(summary)

trgh_ppi_ablation_shs27k = pd.DataFrame([
    {"method": "TRGH-PPI (full)",                "dfs": 72.83, "bfs": 71.05},
    {"method": "w/o Transformer",                 "dfs": 70.44, "bfs": 70.53},
    {"method": "w/o dpdGAT (plain GAT)",           "dfs": 71.42, "bfs": 70.62},
    {"method": "w/ GIN (their structural feats)",  "dfs": 71.92, "bfs": 70.45},
    {"method": "w/o Transformer or dpdGAT",        "dfs": 68.94, "bfs": 67.43},
]).set_index("method")

print("\nTRGH-PPI paper, Table IV (structural features, micro-F1 %):")
print(trgh_ppi_ablation_shs27k)
print("\nOur results, ESM-2 sequence features only (micro-F1 %):")
our_table = (summary["mean"] * 100).unstack("split")[["dfs", "bfs"]].round(2)
print(our_table)